In [3]:
import numpy as np  
import matplotlib.pyplot as plt  
from PIL import Image  
  
def svd_compress_image(image_path, num_singular_values):  
    # 读取图像并转换为 NumPy 数组  
    image = Image.open(image_path).convert('RGB')  
    image_array = np.array(image)[:800,:,:]  
    print(f"image_array.shape:{image_array.shape}")
    # 将图像拆分为三个颜色通道  
    red_channel = image_array[:, :, 0]  
    green_channel = image_array[:, :, 1]  
    blue_channel = image_array[:, :, 2]  
  
    # 对每个通道进行奇异值分解  
    # full_matrices=True:返回完整的矩阵
    U_red, s_red, VT_red = np.linalg.svd(red_channel, full_matrices=False)  
    U_green, s_green, VT_green = np.linalg.svd(green_channel, full_matrices=False)  
    U_blue, s_blue, VT_blue = np.linalg.svd(blue_channel, full_matrices=False)  
    print(f's_red:{s_red.shape}')
    print(f's_green:{s_green.shape}')
    print(f's_blue:{s_blue.shape}')
    print("-="*6)
    print(f"U_red:{U_red.shape}")
    print(f"VT_red:{VT_red.shape}")
    # 仅保留前 num_singular_values 个奇异值和对应的奇异向量  
    U_red = U_red[:, :num_singular_values]  
    s_red = s_red[:num_singular_values]  
    VT_red = VT_red[:num_singular_values, :]  
      
    U_green = U_green[:, :num_singular_values]  
    s_green = s_green[:num_singular_values]  
    VT_green = VT_green[:num_singular_values, :]  
      
    U_blue = U_blue[:, :num_singular_values]  
    s_blue = s_blue[:num_singular_values]  
    VT_blue = VT_blue[:num_singular_values, :]  
  
    # 重构压缩后的图像通道  （dot表示矩阵乘法）
    compressed_red_channel = U_red.dot(np.diag(s_red)).dot(VT_red)  
    compressed_green_channel = U_green.dot(np.diag(s_green)).dot(VT_green)  
    compressed_blue_channel = U_blue.dot(np.diag(s_blue)).dot(VT_blue)  
    
    # 压缩后保存的矩阵
    print(f"compress---U_red.shape:{U_red.shape}")
    print(f"compress---s_red.shape:{s_red.shape}")
    print(f"compress---VT_red.shape:{VT_red.shape}")
    
    # 处理可能出现的浮点数溢出问题（将值限制在 0-255 范围内）  
    compressed_red_channel = np.clip(compressed_red_channel, 0, 255)  
    compressed_green_channel = np.clip(compressed_green_channel, 0, 255)  
    compressed_blue_channel = np.clip(compressed_blue_channel, 0, 255)  
  
    # 将压缩后的通道合并回一个图像  
    compressed_image_array = np.concatenate((compressed_red_channel.astype('uint8')[:,:,None],   
                                        compressed_green_channel.astype('uint8')[:,:,None],   
                                        compressed_blue_channel.astype('uint8')[:,:,None]),
                                             axis=-1)  
    
    # 将 NumPy 数组转换回 PIL 图像并显示  
    compressed_image = Image.fromarray(compressed_image_array)  
    compressed_image.show()  
  
# 使用示例：压缩图像，只保留前 10 个奇异值  
svd_compress_image('w1.jpg', 10)

image_array.shape:(800, 959, 3)
s_red:(800,)
s_green:(800,)
s_blue:(800,)
-=-=-=-=-=-=
U_red:(800, 800)
VT_red:(800, 959)
compress---U_red.shape:(800, 10)
compress---s_red.shape:(10,)
compress---VT_red.shape:(10, 959)


In [1]:
print(f'原图存储的信息：800*959={800*959}')
print(f'svd压缩后的图像存储：800*400+400+400*959={800*400+400+400*959}')

原图存储的信息：800*959=767200
svd压缩后的图像存储：800*400+400+400*959=704000


In [ ]:
# full_matrices=True
U_red:(800, 800)
VT_red:(959, 959)
# full_matrices=False
U_red:(800, 800)
VT_red:(800, 959)